# Context Managers and the `with` Statement

This notebook explores Python's context managers and the `with` statement, providing a comprehensive guide on how to use and create them.

In [ ]:
# Import necessary libraries
import os
import time
import contextlib
import sqlite3
from contextlib import contextmanager, suppress, ExitStack
import threading
import tempfile

## 1. Understanding Resource Management in Python

Resource management is a critical aspect of programming. Resources can include:

- File handles
- Network connections
- Database connections
- Locks and semaphores
- Any object that needs proper acquisition and release

In Python, before context managers, resource management looked like this:

In [ ]:
# Traditional way of managing file resources
def read_file_traditional(filename):
    try:
        file = open(filename, 'r')
        data = file.read()
        return data
    finally:
        # This ensures the file is closed even if an exception occurs
        file.close()

# Example with try-finally for database connection
def query_db_traditional(query):
    connection = sqlite3.connect('example.db')
    cursor = connection.cursor()
    try:
        cursor.execute(query)
        return cursor.fetchall()
    finally:
        cursor.close()
        connection.close()

# Problems with this approach:
# 1. Verbose and repetitive
# 2. Easy to forget closing resources
# 3. Not very Pythonic

## 2. Introduction to Context Managers

Context managers in Python provide a clean and efficient way to manage resources properly. They ensure that setup and teardown operations are performed reliably.

### Key benefits of context managers:

1. Automatic resource cleanup
2. Cleaner, more readable code
3. Exception handling built-in
4. Standardized protocol for resource management

In [ ]:
# Simple context manager example
with open('example.txt', 'w') as f:
    f.write('Hello, Context Managers!')
# File is automatically closed when exiting the with block

# Let's verify the file was created and contains our text
try:
    with open('example.txt', 'r') as f:
        content = f.read()
        print(f"File content: {content}")
except FileNotFoundError:
    print("File doesn't exist yet - run this cell to create it")

## 3. The `with` Statement Syntax

The `with` statement simplifies resource management by providing a block where the resource is guaranteed to be properly managed.

### Syntax:
```python
with context_expression [as target]:
    # Suite of statements where the context is active
```

- `context_expression` is an expression that returns a context manager object
- `as target` (optional) assigns the result of `__enter__()` to the variable `target`
- When the block exits, the context manager's `__exit__()` method is called

In [ ]:
# Basic with statement example
with open('example.txt', 'w') as file:
    file.write('Using the with statement')
    # No need to call file.close()

# Multiple context managers can be used in a single with statement
with open('input.txt', 'w') as input_file, open('output.txt', 'w') as output_file:
    input_file.write('Input data')
    output_file.write('Output data')

# Let's check if the files are accessible outside the with block
try:
    print(file.closed)  # Should be True if file is properly closed
except NameError:
    print("'file' is not accessible outside the with block")
    
# Verifying our files exist
print(f"Files created: input.txt={os.path.exists('input.txt')}, output.txt={os.path.exists('output.txt')}")

## 4. Built-in Context Managers

Python includes several built-in context managers for common tasks:

In [ ]:
# 1. File handling context manager
with open('example.txt', 'w') as f:
    f.write('Example text')

# 2. SQLite database connection context manager
with sqlite3.connect(':memory:') as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE users (name text, age integer)')
    cursor.execute("INSERT INTO users VALUES ('Alice', 25)")
    conn.commit()
    
    # Query the database
    cursor.execute("SELECT * FROM users")
    print(f"Database query result: {cursor.fetchall()}")

# 3. Threading lock context manager
lock = threading.Lock()
with lock:
    # Critical section protected by lock
    print("This section is thread-safe")

# 4. Temporary file context manager
with tempfile.TemporaryFile() as temp:
    temp.write(b'Temporary data')
    temp.seek(0)
    data = temp.read()
    print(f"Temporary file data: {data}")
# File is automatically deleted after the block

# 5. contextlib.suppress - ignores specified exceptions
import os
with suppress(FileNotFoundError):
    os.remove('non_existent_file.txt')  # No exception raised
    print("No exception if file doesn't exist")

## 5. Creating Context Managers Using Classes

To create a custom context manager, you need to implement two magic methods:
1. `__enter__(self)` - Setup code, returns object to be assigned to variable after `as`
2. `__exit__(self, exc_type, exc_val, exc_tb)` - Cleanup code, return value determines exception handling

In [ ]:
# Simple timer context manager using a class
class Timer:
    def __enter__(self):
        self.start = time.time()
        # The return value is assigned to the variable after 'as'
        return self
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        self.end = time.time()
        self.elapsed = self.end - self.start
        print(f"Elapsed time: {self.elapsed:.6f} seconds")
        # Returning False (default) allows exceptions to propagate
        # Returning True would suppress exceptions
        return False
    
# Using our custom context manager
with Timer() as timer:
    # Simulate work
    sum([i**2 for i in range(1000000)])

# A database connection context manager
class DatabaseConnection:
    def __init__(self, db_name):
        self.db_name = db_name
        self.connection = None
    
    def __enter__(self):
        self.connection = sqlite3.connect(self.db_name)
        return self.connection
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        if self.connection:
            if exc_type is None:
                # No exception, commit changes
                self.connection.commit()
            else:
                # Exception occurred, rollback changes
                self.connection.rollback()
                print(f"Database transaction rolled back due to {exc_type.__name__}: {exc_val}")
            
            # Always close the connection
            self.connection.close()
        # Let the exception propagate
        return False

# Using our database context manager
with DatabaseConnection(':memory:') as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE items (name text, quantity integer)')
    cursor.execute("INSERT INTO items VALUES ('apple', 10)")
    cursor.execute("SELECT * FROM items")
    print(f"Database result: {cursor.fetchall()}")

# Example with an exception
try:
    with DatabaseConnection(':memory:') as conn:
        cursor = conn.cursor()
        cursor.execute('CREATE TABLE items (name text, quantity integer)')
        cursor.execute("INSERT INTO items VALUES ('apple', 10)")
        # This will cause an error - table 'non_existent' doesn't exist
        cursor.execute("SELECT * FROM non_existent")
except sqlite3.OperationalError as e:
    print(f"Caught exception: {e}")

## 6. Creating Context Managers Using the `contextlib` Module

The `contextlib` module provides decorators and utilities for creating context managers without defining a full class.

In [ ]:
# Using contextlib.contextmanager decorator
import contextlib

@contextlib.contextmanager
def timer():
    start = time.time()
    try:
        # The yield statement separates setup from cleanup
        # The yielded value is assigned to the variable after 'as'
        yield start
    finally:
        end = time.time()
        print(f"Elapsed time: {end - start:.6f} seconds")

# Using our contextlib-based timer
with timer() as start_time:
    # Simulate work
    sum([i**2 for i in range(500000)])
    print(f"Start time was: {start_time}")

# Database connection using contextlib
@contextlib.contextmanager
def db_connection(db_name):
    conn = sqlite3.connect(db_name)
    try:
        yield conn
        # If we get here without an exception, commit changes
        conn.commit()
    except Exception as e:
        # An exception occurred, rollback changes
        conn.rollback()
        print(f"Database transaction rolled back due to {type(e).__name__}: {e}")
        raise  # Re-raise the exception
    finally:
        # Always close the connection
        conn.close()

# Using our database context manager
with db_connection(':memory:') as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE books (title text, author text)')
    cursor.execute("INSERT INTO books VALUES ('Python Cookbook', 'David Beazley')")
    cursor.execute("SELECT * FROM books")
    print(f"Query result: {cursor.fetchall()}")

# Example of the contextlib.suppress context manager
import os
with contextlib.suppress(FileNotFoundError):
    os.remove('non_existent_file.txt')  # No exception raised
    print("Suppressed FileNotFoundError")

# Example of the contextlib.redirect_stdout context manager
import sys
from io import StringIO

# Redirect stdout to a buffer
buffer = StringIO()
with contextlib.redirect_stdout(buffer):
    print("This goes to the buffer instead of the console")
    
# Now we can get the output from the buffer
output = buffer.getvalue()
print(f"Captured output: {output}")

## 7. Nested Context Managers

Context managers can be nested, allowing for complex resource management scenarios.

In [ ]:
# Basic nesting of context managers
with open('outer.txt', 'w') as outer_file:
    outer_file.write('Outer file content\n')
    
    with open('inner.txt', 'w') as inner_file:
        inner_file.write('Inner file content\n')
        
        # Both files are open here
        outer_file.write('Written while inner file is open\n')
        inner_file.write('Written while outer file is open\n')
    
    # Only outer file is open here
    outer_file.write('Written after inner file is closed\n')

# Alternative syntax for multiple context managers
with open('file1.txt', 'w') as f1, open('file2.txt', 'w') as f2:
    f1.write('File 1 content\n')
    f2.write('File 2 content\n')

# Using ExitStack for dynamic context management
with contextlib.ExitStack() as stack:
    files = []
    # Dynamically create and manage multiple context managers
    for i in range(3):
        filename = f'stack_file_{i}.txt'
        # Each enter_context call adds the context manager to the stack
        # and returns the result of its __enter__ method
        file = stack.enter_context(open(filename, 'w'))
        files.append(file)
    
    # Now we can use all files
    for i, file in enumerate(files):
        file.write(f'Content for file {i}')
    
    print(f"Created {len(files)} files with ExitStack")
# All files are automatically closed when the ExitStack context exits

# Verify our files were created
for i in range(3):
    filename = f'stack_file_{i}.txt'
    if os.path.exists(filename):
        with open(filename, 'r') as f:
            print(f"Content of {filename}: {f.read()}")

## 8. Context Managers for Resource Management

Context managers are particularly useful for managing various types of resources:

In [ ]:
# 1. File locks
@contextmanager
def file_lock(filename):
    # In a real implementation, this would use actual file locking
    lock_file = f"{filename}.lock"
    
    # Acquire the lock
    with open(lock_file, 'w') as f:
        f.write('locked')
    print(f"Lock acquired for {filename}")
    
    try:
        yield  # The context manager doesn't return a value
    finally:
        # Release the lock
        os.remove(lock_file)
        print(f"Lock released for {filename}")

# Using our file lock context manager
with file_lock('important_data.txt'):
    print("Processing important data...")
    # In a real scenario, this is where we'd work with the locked file
    time.sleep(0.5)  # Simulate work

# 2. Temporary directory context manager
@contextmanager
def temp_directory():
    # Create a temporary directory
    dir_path = tempfile.mkdtemp()
    print(f"Created temporary directory: {dir_path}")
    
    try:
        yield dir_path
    finally:
        # Clean up by removing the directory and all its contents
        # In a real implementation, we'd use shutil.rmtree(dir_path)
        print(f"Would remove directory: {dir_path} (not actually removing in this example)")

# Using the temporary directory context manager
with temp_directory() as temp_dir:
    temp_file_path = os.path.join(temp_dir, 'temp_file.txt')
    with open(temp_file_path, 'w') as f:
        f.write('Temporary file in temporary directory')
    print(f"Created file at {temp_file_path}")

# 3. Database transaction context manager
@contextmanager
def transaction(connection):
    cursor = connection.cursor()
    try:
        yield cursor
        connection.commit()
        print("Transaction committed")
    except Exception as e:
        connection.rollback()
        print(f"Transaction rolled back due to {type(e).__name__}: {e}")
        raise

# Using the transaction context manager
conn = sqlite3.connect(':memory:')

# First, create a table
cursor = conn.cursor()
cursor.execute('CREATE TABLE products (id INTEGER PRIMARY KEY, name TEXT, price REAL)')

# Successful transaction
with transaction(conn) as cursor:
    cursor.execute("INSERT INTO products VALUES (1, 'Apple', 1.20)")
    cursor.execute("INSERT INTO products VALUES (2, 'Orange', 0.95)")

# Failed transaction
try:
    with transaction(conn) as cursor:
        cursor.execute("INSERT INTO products VALUES (3, 'Banana', 0.75)")
        # This will fail because 'quantity' column doesn't exist
        cursor.execute("UPDATE products SET quantity = 10 WHERE id = 1")
except sqlite3.OperationalError:
    print("Caught the database error")

# Check what's in the database
cursor = conn.cursor()
cursor.execute("SELECT * FROM products")
print(f"Products in database: {cursor.fetchall()}")
conn.close()

## 9. Error Handling in Context Managers

Context managers provide sophisticated error handling capabilities through the `__exit__` method.

In [ ]:
# Context manager that can handle and suppress specific exceptions
class HandleSpecificExceptions:
    def __init__(self, *exception_types):
        # Store the exception types to handle
        self.exception_types = exception_types
    
    def __enter__(self):
        return self
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        # If the exception type matches what we're looking for, suppress it
        if exc_type is not None and issubclass(exc_type, self.exception_types):
            print(f"Suppressing {exc_type.__name__}: {exc_val}")
            return True  # Suppress the exception
        return False  # Let other exceptions propagate

# Using our exception handler context manager
with HandleSpecificExceptions(ValueError, ZeroDivisionError):
    print("About to divide by zero...")
    result = 1 / 0  # This raises ZeroDivisionError
    print("This line won't execute because of the exception")

print("But execution continues after the with block!")

# Another example with a different exception type
try:
    with HandleSpecificExceptions(ValueError, ZeroDivisionError):
        print("About to raise TypeError...")
        # This exception won't be suppressed because it's not in our list
        raise TypeError("This is a type error")
except TypeError as e:
    print(f"Caught TypeError: {e}")

# Context manager that performs clean up even when an exception occurs
@contextmanager
def robust_file_writer(filename):
    temp_filename = f"{filename}.tmp"
    try:
        # Write to a temporary file first
        with open(temp_filename, 'w') as f:
            print(f"Writing to temporary file: {temp_filename}")
            yield f
        
        # If we get here with no exceptions, rename the temporary file
        print(f"Renaming {temp_filename} to {filename}")
        if os.path.exists(filename):
            os.remove(filename)
        os.rename(temp_filename, filename)
    except Exception as e:
        print(f"Exception occurred: {type(e).__name__}: {e}")
        print(f"Cleaning up temporary file: {temp_filename}")
        if os.path.exists(temp_filename):
            os.remove(temp_filename)
        raise  # Re-raise the exception

# Using our robust file writer
try:
    with robust_file_writer('output_file.txt') as f:
        f.write("This is line 1\n")
        f.write("This is line 2\n")
        # Uncomment to simulate a failure
        # raise ValueError("Simulated failure")
    
    # Verify the file was created
    if os.path.exists('output_file.txt'):
        with open('output_file.txt', 'r') as f:
            print(f"Final file content:\n{f.read()}")
except ValueError as e:
    print(f"Operation failed: {e}")

## 10. Advanced Context Manager Patterns

Here are some advanced patterns and techniques with context managers:

In [ ]:
# 1. Reusable context manager
class ReusableContextManager:
    def __init__(self, name):
        self.name = name
        self.active = False
    
    def __enter__(self):
        if self.active:
            raise RuntimeError(f"Context manager {self.name} is already active")
        self.active = True
        print(f"Entering context: {self.name}")
        return self
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        self.active = False
        print(f"Exiting context: {self.name}")
        return False

# Create a reusable context manager
reusable_cm = ReusableContextManager("Example CM")

# Use it multiple times
with reusable_cm:
    print("Inside first context block")

with reusable_cm:
    print("Inside second context block")

# 2. Context manager that tracks time spent in different code blocks
class TimeTracker:
    def __init__(self):
        self.timings = {}
    
    @contextmanager
    def track(self, name):
        start = time.time()
        try:
            yield
        finally:
            end = time.time()
            elapsed = end - start
            if name in self.timings:
                self.timings[name].append(elapsed)
            else:
                self.timings[name] = [elapsed]
    
    def report(self):
        print("Timing Report:")
        for name, times in self.timings.items():
            avg = sum(times) / len(times)
            print(f"{name}: {len(times)} calls, avg {avg:.6f}s, total {sum(times):.6f}s")

# Using the time tracker
tracker = TimeTracker()

# Track time for different operations
with tracker.track("operation 1"):
    # Simulate work
    sum([i**2 for i in range(100000)])

with tracker.track("operation 2"):
    # Simulate different work
    [i*2 for i in range(500000)]

# Call operation 1 again
with tracker.track("operation 1"):
    sum([i**3 for i in range(50000)])

# Generate the report
tracker.report()

# 3. Context manager factory with customizable behavior
def create_logger(log_level):
    @contextmanager
    def logger(name):
        print(f"{log_level} - {name}: Starting")
        try:
            yield
            print(f"{log_level} - {name}: Completed successfully")
        except Exception as e:
            print(f"{log_level} - {name}: Failed with {type(e).__name__}: {e}")
            raise
    return logger

# Create different loggers
debug_logger = create_logger("DEBUG")
info_logger = create_logger("INFO")

# Use the loggers
with debug_logger("test operation"):
    print("Doing something in debug mode")

with info_logger("important operation"):
    print("Doing something in info mode")

# With an exception
try:
    with info_logger("risky operation"):
        print("About to raise an exception")
        raise ValueError("Something went wrong")
except ValueError:
    print("Exception was logged but still propagated")

## 11. Practical Applications

Let's explore some real-world applications of context managers:

In [ ]:
# 1. Changing directory temporarily
@contextmanager
def change_directory(path):
    original_dir = os.getcwd()
    try:
        os.chdir(path)
        print(f"Changed directory to: {os.getcwd()}")
        yield
    finally:
        os.chdir(original_dir)
        print(f"Restored directory to: {os.getcwd()}")

# Using the temporary directory change
print(f"Current directory: {os.getcwd()}")
with change_directory('/tmp' if os.name != 'nt' else 'C:\\Windows\\Temp'):
    print(f"Inside different directory: {os.getcwd()}")
print(f"Back to original directory: {os.getcwd()}")

# 2. Temporary environment variable modification
@contextmanager
def env_var(name, value):
    # Save old value or note its absence
    old_value = os.environ.get(name)
    old_exists = name in os.environ
    
    # Set new value
    os.environ[name] = value
    print(f"Set environment variable {name}={value}")
    
    try:
        yield
    finally:
        # Restore old state
        if old_exists:
            os.environ[name] = old_value
            print(f"Restored environment variable {name}={old_value}")
        else:
            del os.environ[name]
            print(f"Removed environment variable {name}")

# Testing our environment variable context manager
with env_var('TEST_VAR', 'test_value'):
    print(f"Inside context, TEST_VAR = {os.environ.get('TEST_VAR')}")
print(f"Outside context, TEST_VAR = {os.environ.get('TEST_VAR')}")

# 3. Timer for code profiling
@contextmanager
def profiler(description):
    start = time.time()
    try:
        yield
    finally:
        end = time.time()
        print(f"{description}: {end - start:.6f} seconds")

# Using the profiler
with profiler("List comprehension"):
    result = [i**2 for i in range(1000000)]

with profiler("Generator expression"):
    result = sum(i**2 for i in range(1000000))

# 4. Redirecting stdout/stderr for testing or logging
@contextmanager
def capture_output():
    new_stdout = StringIO()
    new_stderr = StringIO()
    old_stdout = sys.stdout
    old_stderr = sys.stderr
    sys.stdout = new_stdout
    sys.stderr = new_stderr
    try:
        yield new_stdout, new_stderr
    finally:
        sys.stdout = old_stdout
        sys.stderr = old_stderr

# Using the output capture
with capture_output() as (out, err):
    print("This goes to the captured stdout")
    print("This is an error", file=sys.stderr)

print(f"Captured stdout: {out.getvalue()}")
print(f"Captured stderr: {err.getvalue()}")

# 5. Atomic file operations (write all or nothing)
@contextmanager
def atomic_write(filename):
    temp_filename = f"{filename}.{os.getpid()}.tmp"
    try:
        with open(temp_filename, 'w') as f:
            yield f
        # If we get here, the context block completed without an exception
        # So we can safely move the temporary file to the target location
        os.replace(temp_filename, filename)
    except Exception:
        # An exception occurred, remove the temporary file
        if os.path.exists(temp_filename):
            os.unlink(temp_filename)
        raise

# Using atomic write
with atomic_write('important_config.txt') as f:
    f.write("Important configuration data\n")
    f.write("More important settings\n")

# Check the contents
try:
    with open('important_config.txt', 'r') as f:
        print(f"Config file contents:\n{f.read()}")
except FileNotFoundError:
    print("Config file not found (this shouldn't happen)")

# Clean up all the temporary files we created
for filename in [f for f in os.listdir('.') if f.endswith('.txt') or f.endswith('.tmp')]:
    try:
        os.remove(filename)
        print(f"Removed temporary file: {filename}")
    except:
        pass

## Summary

Context managers are a powerful feature in Python that simplify resource management and help write cleaner, more robust code:

1. **Resource Management** - Context managers ensure proper acquisition and release of resources like files, connections, and locks

2. **Error Handling** - They provide automatic cleanup even when exceptions occur

3. **Creating Context Managers** - You can create them with classes (implementing `__enter__` and `__exit__`) or functions (using `@contextmanager`)

4. **Building Blocks** - Python includes built-in context managers and the `contextlib` module for creating custom ones

5. **Practical Uses** - They're essential for file operations, database work, thread synchronization, and many other scenarios

Context managers embody Python's philosophy of having clear, explicit ways to handle common operations, making code more maintainable and less error-prone.